In [1]:
import base64
import json
import re

import numpy as np
import us

import pandas as pd
from pandas.api.typing import NAType

In [2]:
# Load dataset
file_name = '../../data/2018-2025/ncsl_ai_db_2018_2025.csv'
df = pd.read_csv(file_name)

# Rename column names for easier understandability and variable access
df.columns = df.columns.str.lower().str.replace(" ", "_")
df = df.rename(columns={"bill_status": "status", "bill_title": "title", "bill_summary": "summary"})
df = df.fillna("")

# Remove rows with no bill, no legislative session, or data
is_removed = (df.bill_number == "") | df.bill_number.str.match(r"No\b|Not\b")
df = df[~is_removed]

print(df.columns)

Index(['state', 'bill_number', 'title', 'status', 'summary', 'category', 'url',
       'page_year'],
      dtype='str')


In [3]:
# Normalize dashes and remove trailing whitespaces from text columns
for column in [c for c in df.columns if c not in ("url", "page_year")]:
    df[column] = (df[column].astype(str).str.normalize("NFKC")
                  .str.replace("[‐-―−]", "-", regex=True)
                  .str.replace(r"\s+", " ", regex=True)
                  .str.strip())

In [4]:
# Fix capitalization mistake in 'status' column
df["status"] = df.status.replace("To governor", "To Governor")

# Fix special case scraping error for 'bill_number'
df.loc[df.bill_number == "B24-558", "bill_number"] = "B 558"

In [5]:
def url_bill_id(u) -> str | None:
    """Extract bill ID from each bill's url source"""
    if not isinstance(u, str) or not u:
        return None

    # BILL SOURCE --> StateNet
    m = re.search(r"ID:bill:([A-Za-z0-9]+)", u)
    if m:
        return m.group(1)

    # BILL SOURCE --> California leginfo site, ex. CA202120220AB13
    m = re.search(r"bill_id=(\d{8}\d?[A-Z]+\d+)", u)
    if m:
        return "CA" + m.group(1)

    # BILL SOURCE --> Maine legislature site, ex. ME130HP1199 (query parameters can come in any order)
    paper = re.search(r"paper=([A-Z]+\d+)", u)
    snum = re.search(r"snum=(\d+)", u)
    if paper and snum:
        return "ME" + snum.group(1) + paper.group(1)

    # BILL SOURCE --> LexisNexis website (comes in base64 JSON)
    if "lexisnexis.com/opentext/" in u:
        token = u.split("/opentext/")[1].split(".")[0]
        try:
            return json.loads(base64.urlsafe_b64decode(token + "=" * (-len(token) % 4)))["id"]
        except Exception:
            return None

    return None


df["bill_id"] = df.url.map(url_bill_id)

# Bills without a usable link get a fallback id instead of NaN
df["bill_id"] = df.bill_id.fillna(df.state + "_" + df.page_year.astype(str) + "_" + df.bill_number.str.replace(" ", ""))

assert df.bill_id.notna().all()

In [6]:
# Add year each session took place and whether it was a special session
session = pd.to_numeric(df.bill_id.str.extract(r"^[A-Z]{2}(\d{4})")[0])
df["session_year"] = session.fillna(df.page_year).astype(int)

# Sort so "last" always means the latest page, and flag special sessions BEFORE the duplicate listings are dropped
df = df.sort_values(["bill_id", "page_year"])
df["is_special_session"] = (df.bill_number.str.contains("session", case=False)
                            .groupby(df.bill_id).transform("any"))

In [7]:
# Every listing of every bill (status can change between pages) - exported at end
history = df.sort_values(["bill_id", "page_year"]).copy()

In [8]:
# Fill title/category gaps from the latest earlier listing of the same bill that has a value
for c in ["title", "category"]:
    latest = (df.assign(**{c: df[c].replace("", np.nan)})
                .dropna(subset=[c]).groupby("bill_id")[c].last())
    df[c] = df[c].replace("", np.nan).fillna(df.bill_id.map(latest))

# Only keep the final/latest bill listing/status
df = df.drop_duplicates("bill_id", keep="last")

In [9]:
state_mapping = us.states.mapping('name', 'abbr')
state_mapping.pop("Virgin Islands")
state_mapping['U.S. Virgin Islands'] = 'VI'

unmapped_states = set(df.state) - set(state_mapping)
assert not unmapped_states, f"States missing from state_mapping: {unmapped_states}"

df.state = df.state.map(state_mapping)
history["state"] = history.state.map(state_mapping)

In [10]:
# Standardized categories retrieved from NCSL website (some have been renamed over the years)
# ex. 'Education Use' and 'Education/Training'
categories = ['Appropriations', 'Audit', 'Child Pornography', 'Criminal Use', 'Cybersecurity', 'Education Use', 'Education/Training',
              'Effect on Labor/Employment', 'Elections', 'Energy', 'Government Use', 'Health Use', 'Housing', 'Impact Assessment',
              'Judicial Use', 'Notification', 'Oversight/Governance', 'Personhood', 'Private Right of Action', 'Private Sector',
              'Provenance', 'Responsible Use', 'Studies', 'Taxes']

def category_pieces(s) -> list[str]:
    """Raw NCSL category text is a list of labels separated by commas or semicolons"""
    return [p.strip().lower() for p in re.split(r"[;,]", s) if p.strip()]


def clean_categories(s) -> str | NAType:
    """Match labels separately so others aren't captured mistakenly"""
    if pd.isna(s):
        return pd.NA
    pieces = category_pieces(s)
    found = [c for c in categories if any(p == c.lower() for p in pieces)]
    found += [c for c in categories if c not in found and any(c.lower() in p for p in pieces)]
    return ";".join(found) if found else pd.NA


# Every piece of raw category text must match one of the categories above
cat_text = df.category.dropna().map(category_pieces).explode().dropna()
unmatched_categories = cat_text[~cat_text.map(lambda t: any(c.lower() in t for c in categories))]
assert unmatched_categories.empty, f"Unrecognized category text: {unmatched_categories.unique()}"

df["category"] = df.category.map(clean_categories)

print("Remaining bills without categories: ", df.category.isna().sum())

Remaining bills without categories:  76


In [11]:
# Replace curly quotes within 'title' and 'summary' text columns
quotes = str.maketrans({"‘": "'", "’": "'", "“": '"', "”": '"'})

for c in ["title", "summary"]:
    df[c] = df[c].map(lambda s: s.translate(quotes) if isinstance(s, str) else s)

In [12]:
df = df.replace("", np.nan)
history = history.replace("", np.nan)

In [13]:
# Final checks - each bill is unique and no null values
assert df.bill_id.is_unique
assert df.state.notna().all() and df.bill_number.notna().all()
assert df.status.notna().all(), f"{df.status.isna().sum()} bills have no status"
assert set(df.category.dropna().str.split(";").explode()) <= set(categories)

# print("Missing values:\n", df.isnull().sum()[lambda s: s > 0])

In [14]:
# Export - Bill status history
history[["bill_id", "state", "page_year", "status"]].to_csv(
    "../../data/2018-2025/bill_status_history_2018_2025.csv", index=False)

In [15]:
# Reorder column names before export - dropping 'url' in the process
column_order = ["bill_id", "bill_number", "state", "session_year", "status", "category",
                 "title", "summary", "is_special_session", "page_year"]
df = df[column_order]

assert len(column_order) == len(df.columns)

# Export - Entinre cleaned dataset
file_name = '../../data/2018-2025/ncsl_ai_db_2018_2025_clean.csv'
df.to_csv(file_name, index=False)